# Real Data Performance
### Predict perfromance of real 3E cells using machine learning. This code only trains the GP model.
### To compare with the ensemble model, save your predictions yPred_gp and reload them in the Model_Perofrmance_3_10 notebook.

In [1]:
#import libraries
import tensorflow as tf
import gpflow

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
%matplotlib qt
import itertools
import os
import datetime

import scipy.stats
import pickle
import mgzip
import pandas as pd
import importlib

import math
from sklearn.metrics import mean_squared_error
import time
from sklearn.preprocessing import RobustScaler


In [2]:
#parameters

import datetime
date = datetime.date.today().strftime('%Y-%m-%d')
date = "2024-04-15"
dataset_filename = "data110323"

In [3]:
#reload data
xCC = pd.read_pickle(r'Features/' + date + r'/xCC.pkl')
yCC = pd.read_pickle(r'Features/' + date + r'/yCC.pkl')

In [4]:
with open(r'Features/' + date + r'/masterList.pkl', 'rb') as f:
    masterList = pickle.load(f)


totalCells = max(masterList.keys())
masterList

cell_num
1        0.33
2        0.33
3        0.33
4        0.33
5         0.5
6         0.5
7         0.5
8         0.5
9         0.5
10        0.8
11        0.8
12        0.8
13        0.8
14        0.8
15        0.8
16        0.8
17        0.8
18        0.8
19        0.8
20        0.8
21        0.8
22        0.8
23        0.8
24        0.8
25        0.8
26        0.8
27        0.8
28        0.8
29          1
30          1
31          1
32          1
33          1
34          1
35          1
36          1
37          1
38          1
39          1
40          1
41          1
42          1
43          1
44          1
45          1
46          1
47          1
48          1
49          1
50    stepped
51    stepped
52    stepped
53    stepped
54    stepped
55    stepped
Name: C-rate, dtype: object

In [5]:
from functions import TrainTestSplit
importlib.reload(TrainTestSplit)

cell_test = [3,8,15,19,30,31,34,51]

cell_train = np.array([x for x in masterList.keys() if x not in cell_test])

#create train and test sets
xTrain, yTrain, xTest, yTest = TrainTestSplit.train_test_split(xCC, yCC, 'cell', cell_test, masterList)
timeTrain = xTrain['Time']
timeTest = xTest['Time']


print(xTrain.shape, xTest.shape)
print(len(xTrain)/(len(xCC)))

Updated
Training list:  [1, 2, 4, 5, 6, 7, 9, 10, 11, 12, 13, 14, 16, 17, 18, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 32, 33, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 52, 53, 54, 55]
Testing list:  [3, 8, 15, 19, 30, 31, 34, 51]
(7523137, 65) (2162149, 65)
0.7767594059690132


In [6]:
masterList[[val for val in cell_test]] #c-rates for test cells

cell_num
3        0.33
8         0.5
10        0.8
14        0.8
16        0.8
24          1
31          1
32          1
39    stepped
Name: C-rate, dtype: object

In [12]:
#check size of each cell
for cell_num in masterList.keys():
    idx = [xCC.index.get_level_values(0) == cell_num][0]
    print(cell_num, len(xCC[idx]))

1 817123
2 671602
3 814892
4 556644
5 590983
6 153715
7 442828
8 558026
9 390522
10 287534
11 97347
12 286414
13 174056
14 179606
15 94412
16 145957
17 75401
18 11782
19 167791
20 152300
21 63936
22 61210
23 192421
24 152864
25 117360
26 96585
27 58231
28 100956
29 77875
30 193803
31 77576
32 69214
33 17361
34 90883
35 122494
36 14775
37 6118
38 90904
39 204472
40 42697
41 51930
42 14363
43 54378
44 26378
45 5445
46 148463
47 28234
48 29472
49 1255
50 1367
51 315331
52 169854
53 18918
54 22168
55 233782
56 77051


#### Now that we know the important features (refer to other Model_Performance notebook), we can drop columns to obtain only the useful ones.

In [6]:

# use_cols = xTrain.iloc[:, rfe.ranking_ == 1].columns

use_cols = ['Energy', 'Capacity', 'Voltage-8', 'Energy-8', 'Power-8']

xTrain = xTrain[use_cols]
xTest = xTest[use_cols]

## Model Development

In [7]:
#Scale the data. Required for GPR
input_scaler = RobustScaler()

xTrain_scaled = xTrain.copy()
xTrain_minmax = input_scaler.fit_transform(xTrain)
xTrain_scaled.values[:]  = xTrain_minmax
xTest_minmax = input_scaler.transform(xTest)

xTrain_minmax.shape

(7523137, 5)

### GP Model

In [8]:
#model setup

inputDim =xTrain_minmax.shape[1]
outputDim = 1

var_val = np.sqrt(0.002**2 / len(xTrain))

#kernel setup 
k = gpflow.kernels.Matern52(lengthscales= np.ones(inputDim)) + gpflow.kernels.Constant() + gpflow.kernels.White(variance=var_val)

#inducing points
num_points = 800

base = 15
inc = base * round(np.ceil(xTrain_minmax.shape[0]/num_points)/base)

Z = xTrain_minmax[:num_points*base:base, :].copy()

#model setup
m = gpflow.models.SVGP(kernel=k, likelihood= gpflow.likelihoods.Gaussian(),inducing_variable=Z, num_data = xTrain_minmax.shape[0])


data = (xTrain_minmax, yTrain)

elbo = tf.function(m.elbo)



In [9]:
#model training

from gpflow.ci_utils import ci_niter

import gpflow.optimizers

def run_adam(model, iterations):
    """
    Utility function running the Adam optimizer

    :param model: GPflow model
    :param interations: number of iterations
    """
    # Create an Adam Optimizer action
    logf = []
    train_iter = iter(train_dataset.batch(minibatch_size))
    training_loss = model.training_loss_closure(train_iter, compile=True)
    optimizer = tf.optimizers.Adam(learning_rate=0.001)

    @tf.function
    def optimization_step():
        optimizer.minimize(training_loss, model.trainable_variables)

    for step in range(iterations):
        optimization_step()
        if step % 10 == 0:
            elbo = -training_loss().numpy()
            logf.append(elbo)
    return logf

# Create a tf.data dataset from the numpy arrays
train_dataset = tf.data.Dataset.from_tensor_slices((xTrain_minmax, pd.DataFrame(yTrain).to_numpy())).repeat().shuffle(len(yTrain))

minibatch_size = 16

maxiter = ci_niter(150)

# Run the Adam optimizer
logf = run_adam(m, maxiter)

params = gpflow.utilities.parameter_dict(m)

#save data
if not os.path.exists('models/' + date + '/' + dataset_filename):
    os.makedirs('models/' + date + '/' + dataset_filename)

with open('models/' + date + '/' + dataset_filename + '/svgp_model.pkl',"wb") as f:
    pickle.dump(params,f)



In [10]:
#check training metric. ELBO should increase then stabilize

plt.figure()
plt.plot(np.arange(maxiter)[::10], logf)
plt.xlabel('Number of iterations')
plt.ylabel('ELBO')


Text(0, 0.5, 'ELBO')

In [11]:
#make predictions

val = len(xTest_minmax)/10

yPred_gp = [] 
yStd = []
for i in range(10):
    start = int(val*i)
    end = int(val*(i+1))
    yPred_temp, yStd_temp = m.predict_f(xTest_minmax[start:end,:])
    yPred_gp.append(yPred_temp)
    yStd.append(yStd_temp)
yPred_gp = np.array(np.concatenate(yPred_gp)).flatten()
yExp = yTest[::].to_numpy().flatten()
yStd = 1.96 * np.sqrt(np.concatenate(yStd))

yPred = yPred_gp

np.savetxt('models/' + date + '/' + dataset_filename + 'svgp_results.npy', yPred_gp)

### Reload model. Make sure you have set up the model itself.

In [ ]:
try:
    with open('models/' + date + '/' + dataset_filename + '/svgp_model.pkl', 'rb') as f:
        params = pickle.load(f)

    gpflow.utilities.multiple_assign(m, params)
except Exception as e:
    print(e)
    
yPred_gp = np.load('models/' + date + '/' + dataset_filename + 'svgp_results.npy')

### Baseline linear regression, if you want to compare in this notebook.

In [ ]:
from sklearn.linear_model import LinearRegression

baseline = LinearRegression()
reg = baseline.fit(xTrain_minmax, yTrain)
print(reg.coef_)
yPred_baseline = reg.predict(xTest_minmax)


filename = open('models/' + date + '/' + dataset_filename + '/baseline.pkl', 'wb')
pickle.dump(reg, filename)
filename.close()

[-2.81031503e-01  2.66525291e-01  2.81957351e-01 -3.16397857e-01
  1.83391353e-04]


## Evaluate Model Performance

In [ ]:
##linear regression for observing relationship between predicted and actual data
slope, intercept, r, p, stderr = scipy.stats.linregress(yPred.T, yExp.T)
line = f'Regression line: y={intercept:.2f}+{slope:.2f}x, r={r:.2f}'

plt.figure()
plt.plot(yPred, yExp, '*', label='data', markersize = 0.1)
plt.plot(yPred, intercept + slope * yPred, label=line)

plt.legend()
plt.xlabel('Predicted NEP')
plt.ylabel('Actual NEP')

ax = plt.gca()
ax.tick_params(axis = 'both', which = 'major', labelsize = 15)
ax.tick_params(axis = 'both', which = 'minor', labelsize = 15)

In [ ]:

#specify which model to evaluate metrics for
yPred = yPred_gp

In [ ]:
from sklearn.metrics import mean_squared_error

#Get error metrics
mpe = abs(yPred - yExp)
mae = mpe.mean()

rmse = mean_squared_error(yExp, yPred, squared=False)
rmspe = np.sqrt(np.mean(np.square(((yExp - yPred) / abs(yExp))), axis=0)) * 100

print('MAE:', "{:.5f}".format(mae), 'V')
print('RMSE:', "{:.5f}".format(rmse), 'V')

#plot absolute differences
plt.figure()
plt.plot(yExp, abs(yExp - yPred), 'r*',markersize = 0.7)
plt.xlabel('Actual NEP', fontsize=12)
plt.ylabel('Absolute NEP Difference', fontsize=12)

ax = plt.gca()
ax.tick_params(axis = 'both', which = 'major', labelsize = 12)
ax.tick_params(axis = 'both', which = 'minor', labelsize = 12)

MAE: 0.01553 V
RMSE: 0.02018 V


In [ ]:
from sklearn.metrics import mean_squared_error

#Get error metrics
mpe = abs(yPred - yExp)
mae = mpe.mean()

rmse = mean_squared_error(yExp, yPred, squared=False)
rmspe = np.sqrt(np.mean(np.square(((yExp - yPred) / abs(yExp))), axis=0)) * 100

print('MAE:', "{:.5f}".format(mae), 'V')
print('RMSE:', "{:.5f}".format(rmse), 'V')

#plot absolute differences
plt.figure()
plt.plot(yExp, abs(yExp - yPred), 'r*',markersize = 0.7)
plt.xlabel('Actual NEP', fontsize=12)
plt.ylabel('Absolute NEP Difference', fontsize=12)
ax = plt.gca()
ax.tick_params(axis = 'both', which = 'major', labelsize = 12)
ax.tick_params(axis = 'both', which = 'minor', labelsize = 12)

print('Averages across cells')
#Get error metrics
mae_list = []
rmse_list = []
for cell_num in cell_test:
    idx = [xTest.index.get_level_values(0) == cell_num][0]
    mae = abs(yPred[idx] - yExp[idx]).mean()
    rmse = mean_squared_error(yExp[idx], yPred[idx], squared=False)
    print('Dataset size: ', len(yPred[idx]) )
    print('MAE:', "{:.5f}".format(mae), 'V')
    print('RMSE:', "{:.5f}".format(rmse), 'V')



MAE: 0.01418 V
RMSE: 0.01928 V
Averages across cells
Dataset size:  519811
MAE: 0.00731 V
RMSE: 0.00949 V
Dataset size:  442995
MAE: 0.02016 V
RMSE: 0.02483 V
Dataset size:  44364
MAE: 0.01883 V
RMSE: 0.02451 V
Dataset size:  82301
MAE: 0.01850 V
RMSE: 0.02341 V
Dataset size:  69000
MAE: 0.01367 V
RMSE: 0.01888 V
Dataset size:  34876
MAE: 0.01766 V
RMSE: 0.02142 V
Dataset size:  20160
MAE: 0.01215 V
RMSE: 0.01544 V
Dataset size:  71729
MAE: 0.01864 V
RMSE: 0.02383 V


In [ ]:
#plot predicted vs actual data
from scipy.stats import pearsonr
yPred_list = {'baseline': yPred_baseline, 'gp': yPred_gp }
select_cell = 4

for select_cell in range(len(cell_test)):
    indList = timeTest.index.get_loc(tuple([cell_test[select_cell]]))
    for key in yPred_list:
        plt.figure(figsize=(7.04, 5.28))
        corr, _ = pearsonr(yExp[indList], yPred_list[key][indList])
        corr *=100
        plt.plot(yExp[indList], yPred_list[key][indList], '*', label='data, r^2 = %.2f' % corr, markersize = 0.7)
        plt.plot(yExp[indList], yExp[indList])
        
        plt.legend()
        plt.xlabel('Observed NEP')
        plt.ylabel('Predicted NEP')
        ax = plt.gca()
        
        plt.suptitle('Cell ' + str(cell_test[select_cell]) + ' ' + key)
        plt.savefig('models/' + date + '/' + dataset_filename + '/linear_regression_' + key + '__cell_' + str(select_cell))
        

In [ ]:
#error of individual cycles

plt.figure()

yPred_list = {'baseline': yPred_baseline, 'gp': yPred_gp }
cycle_error = {'baseline': [],'gp': []}

select_cell = 0

maxCycles = xTest.loc[cell_test[select_cell]].index.max()

for key, value in yPred_list.items():
    for cycle in range(1, maxCycles, 1):
        try:
            indList = timeTest.index.get_loc(tuple([cell_test[select_cell], cycle])) 
            cycle_error[key].append(mean_squared_error(yExp[indList], value[indList], squared=False))
        except:
            # print(key)
            cycle_error[key].append(np.nan)
    plt.plot(np.arange(1, maxCycles, 1),  cycle_error[key], '*-', linewidth = 2, label = key)


ax = plt.gca()
ax.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(nbins = 10, integer=True))
plt.xlabel('Cycle')
plt.ylabel('RMSE [V]')
plt.legend(prop = {'size':10})
plt.savefig('models/' + date + '/' + dataset_filename + '/individual_cycle_error_' + str(select_cell))